# TransformerPayne Integration — documentation examples

Companion notebook to the **TransformerPayne Integration** documentation page
(`docs/source/transformer_payne_integration.rst`). One section per code
snippet / figure; the figure sections regenerate `docs/img/tpayne_*`,
`mn_spot_*`, and `mn_line_profile*` (light and dark variants).

Requires the `transformer-payne` and `huggingface-hub` packages
(`pip install stellar-spice[huggingface] transformer-payne`); the model
download is ~several hundred MB, so this notebook is shipped unexecuted.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")

from pathlib import Path
import jax.numpy as jnp
import matplotlib.pyplot as plt

# Figures are written straight into the documentation image directory with the
# exact filenames the .rst pages reference.
DOCS_IMG = Path("..") / ".." / "docs" / "img"

def save_doc_fig(name, make_fig):
    """Render ``make_fig()`` twice — default (light) style and dark style — and
    save as <name>.png / <name>_dark.png in docs/img."""
    for style, suffix in [("default", ""), ("dark_background", "_dark")]:
        with plt.style.context(style):
            fig = make_fig()
            fig.savefig(DOCS_IMG / f"{name}{suffix}.png", dpi=120, bbox_inches="tight",
                        facecolor=fig.get_facecolor())
            plt.close(fig)
    print(f"saved {name}.png / {name}_dark.png")

import numpy as np
from tqdm import tqdm

## Downloading TransformerPayne

In [ ]:
from transformer_payne import TransformerPayne

tp = TransformerPayne.download()

## Creating a Mesh Model

TransformerPayne expects temperature as log10 (`logteff`), not linear `teff`.

In [ ]:
from spice.models import IcosphereModel

m = IcosphereModel.construct(1000, 1., 1.,
                             tp.to_parameters(dict(logteff=jnp.log10(7000), logg=4.3, O=8.0, Si=6.0)),
                             tp.stellar_parameter_names)

## Spectrum of a rotating star

Figure: `tpayne_spectrum_rotation.png` / `tpayne_spectrum_rotation_dark.png`

In [ ]:
from spice.models.mesh_transform import add_rotation, evaluate_rotation
from spice.spectrum import simulate_observed_flux

mt = add_rotation(m, 100, jnp.array([0., 0., 1.]))
mt = evaluate_rotation(mt, 0.)

vws = np.linspace(4670, 4960, 2000)
spec_no_rot = simulate_observed_flux(tp.intensity, m, jnp.log10(vws))
spec_rot = simulate_observed_flux(tp.intensity, mt, jnp.log10(vws))

def rotation_spectrum_fig():
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(vws, spec_no_rot[:, 0], color='black', linewidth=1, label='No rotation')
    ax.plot(vws, spec_rot[:, 0], color='royalblue', linewidth=3, label='100 km/s')
    ax.set_xlabel(r'Wavelength [$\AA$]')
    ax.set_ylabel(r'Normalized Flux [erg/s/cm$^2$/$\AA$]')
    ax.legend()
    return fig

save_doc_fig("tpayne_spectrum_rotation", rotation_spectrum_fig)

## A rotating star with a manganese spot

Figures: `mn_spot_0.png` and `mn_spot_50.png` (+`_dark` variants)

In [ ]:
from spice.models.spots import add_spot
from spice.plots import plot_3D

timestamps = np.linspace(0, 48*3600, 100)

m_spotted = add_spot(m, spot_center_theta=1., spot_center_phi=1., spot_radius=30.,
                     parameter_delta=5.0, parameter_index=tp.parameter_names.index('Mn'))
m_spotted = [evaluate_rotation(add_rotation(m_spotted, 25.), t) for t in timestamps]

mn_index = tp.parameter_names.index('Mn')
save_doc_fig("mn_spot_0",
             lambda: plot_3D(m_spotted[0], property_label='Mn abundance', property=mn_index)[0])
save_doc_fig("mn_spot_50",
             lambda: plot_3D(m_spotted[50], property_label='Mn abundance', property=mn_index)[0])

## Line profiles across the rotation

Figure: `mn_line_profile.png` / `mn_line_profile_dark.png`

In [ ]:
vws = np.linspace(4762, 4769, 2000)
spec_rot_spotted = [simulate_observed_flux(tp.intensity, _m_spotted, jnp.log10(vws))
                    for _m_spotted in tqdm(m_spotted)]

def line_profile_fig():
    fig, ax = plt.subplots(figsize=(12, 6))
    colors = plt.cm.cool(np.linspace(0, 1, len(spec_rot_spotted)))
    for i, spectrum in enumerate(spec_rot_spotted):
        ax.plot(vws, spectrum[:, 0], color=colors[i], linewidth=1, alpha=0.5)
    sm = plt.cm.ScalarMappable(cmap=plt.cm.cool,
                               norm=plt.Normalize(vmin=0, vmax=timestamps[-1]/(3600)))
    fig.colorbar(sm, ax=ax, label='Time [h]')
    ax.set_xlabel(r'Wavelength [$\AA$]')
    ax.set_ylabel(r'Flux [erg/s/cm$^2$/$\AA$]')
    return fig

save_doc_fig("mn_line_profile", line_profile_fig)

## Line profiles of a pulsating star

Figure: `tpayne_pulsation.png` / `tpayne_pulsation_dark.png`

In [ ]:
from spice.models.mesh_transform import add_pulsation, evaluate_pulsations
import cmasher as cmr

m = IcosphereModel.construct(5000, 1., 1.,
                             tp.to_parameters(dict(logteff=np.log10(8340), logg=4.3)),
                             tp.stellar_parameter_names)
mp = add_pulsation(m, 0, 0, 5., jnp.array([[1e-4, 0.]]))

TIMESTAMPS = jnp.linspace(0., 5., 20)

mps = [evaluate_pulsations(mp, t) for t in tqdm(TIMESTAMPS)]

vws = np.linspace(4762, 4769, 2000)
specs = [simulate_observed_flux(tp.intensity, _m_pulsating, jnp.log10(vws))
         for _m_pulsating in tqdm(mps)]

def pulsation_profiles_fig():
    cmap = cmr.bubblegum
    norm = plt.Normalize(TIMESTAMPS.min(), TIMESTAMPS.max())
    fig, ax = plt.subplots()
    for spec, timestamp in zip(specs, TIMESTAMPS):
        ax.plot(vws, spec[:, 0], color=cmap(norm(timestamp)))
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, ticks=TIMESTAMPS)
    cbar.set_label('Time [d]')
    cbar.set_ticklabels([f'{t:.2f}' for t in TIMESTAMPS])
    ax.set_xlabel(r'Wavelength [$\AA$]')
    ax.set_ylabel(r'Intensity [erg/s/cm$^2$/$\AA$]')
    ax.tick_params(axis='x', rotation=45)
    return fig

save_doc_fig("tpayne_pulsation", pulsation_profiles_fig)